### CacheBackedEmbeddings
- 임베딩을 한 번 수행한 후, 결과를 파일 형태로 저자하는 방식 = 캐시파일

In [3]:
from langchain_classic.storage import LocalFileStore
from langchain_openai import OpenAIEmbeddings
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_community.vectorstores.faiss import FAISS
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("test0914")

embedding = OpenAIEmbeddings()

store = LocalFileStore("./cache/")

LangSmith 추적을 시작합니다.
[프로젝트명]
test0914


In [4]:
cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings= embedding, # 임베딩 모델 지정
    document_embedding_cache= store,  # ./cache/ 폴더
    namespace= embedding.model   # 임베딩 값 구분자
)

d:\kingSJ\hanwha_0902\ex_0922\.venv\Lib\site-packages\langchain_classic\embeddings\cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()


In [ ]:
list(store.yield_keys()) # 아직 캐싱된 파일이 없으므로 빈 리스트

[]

In [6]:
from langchain_classic.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

raw_documents = TextLoader("appendix-keywords.txt", encoding="utf-8").load()
text_splitter = CharacterTextSplitter(chunk_size = 1000, chunk_overlap = 0)
documents = text_splitter.split_documents(raw_documents)

In [7]:
%time db = FAISS.from_documents(documents, cached_embedder)

CPU times: total: 234 ms
Wall time: 2.29 s


In [8]:
%time db2 = FAISS.from_documents(documents, cached_embedder) # 다시 생성하려면 하면 임베딩을 훨씬 빠르게 처리됨

CPU times: total: 31.2 ms
Wall time: 60.9 ms


In [9]:
from langchain_classic.storage import InMemoryByteStore

store = InMemoryByteStore()

# 캐시 지원 임베딩 생성
cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    embedding, store, namespace=embedding.model
)